## Assignment 3: Search and Games

Welcome to the third assignment of CS 541! In Assignments 1 and 2 the machine learned from data. Here nothing is learned: the agent *searches*. You will play tic-tac-toe against a rule-based bot, and then build a sequence of stronger players, from a one-move greedy agent up to full minimax with alpha-beta pruning, until you have one that provably never loses. This assignment must be done individually.

After this assignment, you should be able to:
1. Formalize a two-player game as a search problem, and evaluate agents by playing matches between them.  
2. Implement a greedy agent, minimax, alpha-beta pruning, and depth-limited search with an evaluation function, and measure what each costs.  

In CS541, any work generated by an AI should be declared. For this submission, **ChatGPT was used as a study, coding-review, and proofreading assistant** to help interpret the assignment and lecture requirements, check implementation logic, identify grading-sensitive edge cases, and polish the explanations/comments. The final notebook was reviewed and executed against the starter specifications; no course instructions or function signatures were changed.

Nothing in this assignment is downloaded. Part 1 uses `ipywidgets` to draw a clickable board; if the import below fails, run `pip install ipywidgets` in your environment and restart the kernel. Everything else is plain Python plus numpy, pandas and matplotlib. Run the cell below first.

In [29]:
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


### 1. Play the game (6')
Tic-tac-toe is written below exactly as the lecture formalizes a game: an initial state $S_0$, $\text{Player}(s)$, $\text{Actions}(s)$, $\text{Result}(s, a)$, $\text{Terminal-Test}(s)$ and $\text{Utility}(s, p)$. A state is a tuple of 9 characters, `"X"`, `"O"` or `"."`, with cells numbered 0 to 8 row by row. X always moves first. An **agent** is any function that takes a state and returns the cell it wants to play. Everything in this part and the next two is built from that one interface.

Two agents are given. `RandomAgent(seed)` plays uniformly at random among the legal moves, using its own seeded generator, so a match against it is reproducible. `rule_based_agent` follows five rules in order: (1) win now if it can; (2) otherwise block the opponent's immediate win; (3) otherwise take the center; (4) otherwise take the first free corner in the order 0, 2, 6, 8; (5) otherwise take the first free edge in the order 1, 3, 5, 7. It is deterministic.

1.1 Complete `play_game()` and `run_match()` below.  
1.2 Play the bot with the widget: at least **five** complete games, at least **two** as X and at least **two** as O. The widget prints the full move list of every game when it ends. Paste those lists into `my_games`, one list per game.  
1.3 Find a sequence of your own moves, as X, that beats the bot. Store *your moves only* as `beat_the_bot_moves`. Then explain which of the bot's five rules let it down, and why no rule of that form could have saved it.  
1.4 Run three baseline matches of 1000 games each and store the result dicts as `match_random_random` (`RandomAgent(0)` as X, `RandomAgent(1)` as O), `match_rule_vs_random` (`rule_based_agent` as X, `RandomAgent(1)` as O) and `match_random_vs_rule` (`RandomAgent(0)` as X, `rule_based_agent` as O). Comment on your observations. The bot loses a handful of games as O to a *random* player: find one of those games and say which rule failed.  

Following are some specifications that may be helpful:
- `play_game(agent_x, agent_o)` always starts from `initial_state()`, asks the agent whose turn it is for exactly one move per ply, and returns `(winner, moves)`, where `winner` is `"X"`, `"O"` or `None` for a draw, and `moves` is the list of every cell played, in order.  
- `run_match(agent_x, agent_o, n_games)` returns a dict with exactly the keys `"X"`, `"O"` and `"draw"`, counting games.  
- Create each `RandomAgent` once, right before its match, and pass the same object into all of that match's games. Its generator then advances from game to game, and the whole match is reproducible.  
- Note: don't modify the given engine cell. Every later part, and the grader, relies on it behaving exactly as written.  

In [30]:
# Given -- do not modify. The game, in the lecture's formalization.
EMPTY = "."
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
         (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
         (0, 4, 8), (2, 4, 6)]              # diagonals


def initial_state():
    """S_0: the empty board."""
    return (EMPTY,) * 9


def player(s):
    """Player(s): X moves first, so it is X's turn whenever the counts are equal."""
    return "X" if s.count("X") == s.count("O") else "O"


def other(p):
    return "O" if p == "X" else "X"


def actions(s):
    """Actions(s): the empty cells, in increasing order."""
    return [i for i, c in enumerate(s) if c == EMPTY]


def result(s, a):
    """Result(s, a): a NEW state with the current player's mark on cell a."""
    if s[a] != EMPTY:
        raise ValueError("cell {} is already taken".format(a))
    return s[:a] + (player(s),) + s[a + 1:]


def winner(s):
    """'X' or 'O' if that player has three in a line, else None."""
    for i, j, k in LINES:
        if s[i] != EMPTY and s[i] == s[j] == s[k]:
            return s[i]
    return None


def terminal(s):
    """Terminal-Test(s): someone has won, or the board is full."""
    return winner(s) is not None or EMPTY not in s


def utility(s, p):
    """Utility(s, p) at a terminal state: +1 if p won, -1 if p lost, 0 for a draw."""
    w = winner(s)
    return 0 if w is None else (1 if w == p else -1)


def show_board(s):
    print("\n".join(" " + " | ".join(s[r * 3:r * 3 + 3]) for r in range(3)))


def all_reachable_states():
    """Every board reachable from the empty board by legal play, terminal ones included."""
    seen, stack = set(), [initial_state()]
    while stack:
        s = stack.pop()
        if s in seen:
            continue
        seen.add(s)
        if not terminal(s):
            stack.extend(result(s, a) for a in actions(s))
    return seen


class RandomAgent:
    """Plays uniformly at random among the legal moves, from its own seeded generator."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def __call__(self, s):
        acts = actions(s)
        return acts[self.rng.integers(len(acts))]


CORNERS, EDGES = (0, 2, 6, 8), (1, 3, 5, 7)


def rule_based_agent(s):
    """Win if possible, else block, else center, else first free corner, else first free edge."""
    me, opp = player(s), other(player(s))
    acts = actions(s)
    for a in acts:                                   # rule 1: win now
        if winner(result(s, a)) == me:
            return a
    for a in acts:                                   # rule 2: block the opponent's win
        if winner(s[:a] + (opp,) + s[a + 1:]) == opp:
            return a
    if s[4] == EMPTY:                                # rule 3: center
        return 4
    for a in CORNERS + EDGES:                        # rules 4 and 5: corners, then edges
        if s[a] == EMPTY:
            return a


print(len(all_reachable_states()), "reachable boards")
show_board(result(result(initial_state(), 4), 0))

5478 reachable boards
 O | . | .
 . | X | .
 . | . | .


In [31]:
# Starter functions completed below. The given game engine in the previous cell is unchanged.
def play_game(agent_x, agent_o):
    """Play one game from the empty board. Returns (winner, moves)."""
    state = initial_state()
    moves = []

    # Continue one ply at a time until the game reaches a terminal state.
    while not terminal(state):
        # Choose the agent for the player whose turn it is.
        agent = agent_x if player(state) == "X" else agent_o
        action = agent(state)

        # Apply the move through the given Result(s, a) function so the
        # exact game rules and illegal-move checks stay centralized.
        state = result(state, action)
        moves.append(action)

    # winner(state) is X/O for a win and None for a draw.
    return winner(state), moves


def run_match(agent_x, agent_o, n_games):
    """Play n_games and return {"X": x_wins, "O": o_wins, "draw": draws}."""
    counts = {"X": 0, "O": 0, "draw": 0}

    # Reuse the same agent objects for all games in this match. In particular,
    # a RandomAgent's seeded generator is meant to advance from game to game.
    for _ in range(n_games):
        game_winner, _ = play_game(agent_x, agent_o)
        if game_winner is None:
            counts["draw"] += 1
        else:
            counts[game_winner] += 1

    return counts


# A quick deterministic sanity check: two copies of the rule-based bot should
# always produce the same complete game.
print(play_game(rule_based_agent, rule_based_agent))


(None, [4, 0, 2, 6, 3, 5, 8, 1, 7])


#### The widget
Run the cell below and click a square to move; the bot answers immediately. Use the two buttons on top to start a new game as X or as O. When a game ends, its full move list (your moves and the bot's) is printed under the board: copy those lists into `my_games` in the next cell. The lists are only printed, never saved, so copy them before you restart the kernel.

In [32]:
# Given -- do not modify.
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run `pip install ipywidgets`, then restart the kernel.")


class TicTacToeWidget:
    """A clickable board against `bot`. Every finished game is printed and kept in self.history."""
    def __init__(self, bot):
        self.bot, self.history = bot, []
        cell = widgets.Layout(width="64px", height="64px")
        self.buttons = [widgets.Button(description=" ", layout=cell) for _ in range(9)]
        for i, b in enumerate(self.buttons):
            b.style.font_weight = "bold"
            b.on_click(lambda _, i=i: self._human_move(i))
        as_x = widgets.Button(description="New game: I play X")
        as_o = widgets.Button(description="New game: I play O")
        as_x.on_click(lambda _: self.new_game("X"))
        as_o.on_click(lambda _: self.new_game("O"))
        grid = widgets.GridBox(self.buttons, layout=widgets.Layout(
            grid_template_columns="repeat(3, 64px)", grid_gap="4px"))
        self.status, self.log = widgets.HTML(), widgets.Output()
        self.ui = widgets.VBox([widgets.HBox([as_x, as_o]), grid, self.status, self.log])
        self.new_game("X")

    def new_game(self, human):
        self.human, self.state, self.moves = human, initial_state(), []
        if human == "O":
            self._apply(self.bot(self.state))
        self._render()

    def _apply(self, a):
        self.state = result(self.state, a)
        self.moves.append(a)

    def _human_move(self, i):
        if terminal(self.state) or player(self.state) != self.human or self.state[i] != EMPTY:
            return
        self._apply(i)
        if not terminal(self.state):
            self._apply(self.bot(self.state))
        self._render()

    def _render(self):
        over = terminal(self.state)
        for i, b in enumerate(self.buttons):
            b.description = " " if self.state[i] == EMPTY else self.state[i]
            b.disabled = over or self.state[i] != EMPTY
        if not over:
            self.status.value = "You are <b>{}</b>. Your move.".format(self.human)
            return
        w = winner(self.state)
        outcome = "draw" if w is None else ("you win" if w == self.human else "the bot wins")
        self.status.value = "<b>Game over: {}.</b>".format(outcome)
        self.history.append(list(self.moves))
        with self.log:
            print("you played {}, {}: {}".format(self.human, outcome, self.moves))


if widgets is not None:
    game = TicTacToeWidget(rule_based_agent)
    display(game.ui)

In [33]:
# Five complete game transcripts using the fixed engine.
# The list contains every move in chronological order, exactly as the widget records it.
# There are at least two games played as X and at least two played as O.
my_games = [
    [0, 4, 7, 2, 6, 3, 8],          # human X
    [0, 4, 1, 2, 3, 6],              # human X
    [8, 4, 7, 6, 5, 2],              # human X
    [4, 0, 2, 1, 6],                  # human O
    [4, 8, 0, 7, 6, 5, 2],            # human O
]

# Your moves only, as X, in a game that beats the deterministic rule-based bot.
# O's replies are omitted here because the assignment asks for X's moves only.
beat_the_bot_moves = [0, 7, 6, 8]

# Each match uses one seeded RandomAgent object per side, reused across all 1000 games.
random_x = RandomAgent(0)
random_o = RandomAgent(1)
match_random_random = run_match(random_x, random_o, 1000)

rule_x = rule_based_agent
random_o = RandomAgent(1)
match_rule_vs_random = run_match(rule_x, random_o, 1000)

random_x = RandomAgent(0)
rule_o = rule_based_agent
match_random_vs_rule = run_match(random_x, rule_o, 1000)

# Compact display of the three required 1000-game baselines.
baseline_results = pd.DataFrame({
    "match": ["random_vs_random", "rule_vs_random", "random_vs_rule"],
    "X": [match_random_random["X"], match_rule_vs_random["X"], match_random_vs_rule["X"]],
    "O": [match_random_random["O"], match_rule_vs_random["O"], match_random_vs_rule["O"]],
    "draw": [match_random_random["draw"], match_rule_vs_random["draw"], match_random_vs_rule["draw"]],
})
display(baseline_results)
print("Required match dictionaries:")
print("match_random_random =", match_random_random)
print("match_rule_vs_random =", match_rule_vs_random)
print("match_random_vs_rule =", match_random_vs_rule)


,match,X,O,draw
0,random_vs_random,590,279,131
1,rule_vs_random,965,0,35
2,random_vs_rule,13,852,135


Required match dictionaries:
match_random_random = {'X': 590, 'O': 279, 'draw': 131}
match_rule_vs_random = {'X': 965, 'O': 0, 'draw': 35}
match_random_vs_rule = {'X': 13, 'O': 852, 'draw': 135}


**1.3 - Why the bot loses to `beat_the_bot_moves = [0, 7, 6, 8]`**

The resulting move sequence is `X0, O4, X7, O2, X6, O3, X8`, so X wins on the bottom row. The bot's **rule 2 (block the opponent's immediate win)** is the point of failure. After X has played cells 6 and 7, X has two separate immediate winning moves, cells 3 and 8: cell 3 completes column 0-3-6, while cell 8 completes row 6-7-8. The bot can block only one threat, so no single move under rule 2 can prevent the loss. This is the classic fork idea: a one-threat blocking rule cannot defeat two simultaneous threats.

**1.4 - Baseline observations**

`random_vs_random` gives `{'X': 590, 'O': 279, 'draw': 131}`. `rule_based_agent` is extremely strong as X against a random O (`{'X': 965, 'O': 0, 'draw': 35}`), but it is not unbeatable as O: random X wins 13 of 1000 games (`{'X': 13, 'O': 852, 'draw': 135}`). The illustrative losing game is `[5, 4, 6, 0, 8, 2, 7]`. After X plays 5, 6, and 8, X threatens both 7 and 2. O's rule-2 block at cell 2 removes one threat but cannot remove the other, so X wins at 7.


### 2. A greedy agent (7')
The rule-based bot hard-codes what to do. A *greedy* agent is the simplest thing that searches instead: it tries each legal move, scores the board it would produce with an evaluation function, and plays the best one. This is greedy best-first search from Lecture 4, applied to a game: it ranks moves by a heuristic alone and never looks at what happens next.
2.1 Implement `evaluate(s, p)`, the line-counting evaluation function defined below.  
2.2 Implement `greedy_agent(s)`.  
2.3 Build a dict named `greedy_results` with exactly the keys `"greedy_vs_random"`, `"random_vs_greedy"`, `"greedy_vs_rule"` and `"rule_vs_greedy"`, each holding a `run_match` result. The first-named agent plays X. Use 1000 games and `RandomAgent(1)` as O / `RandomAgent(0)` as X for the random matches, and a single game for the two matches against the bot. Briefly explain why one game is enough there.  
2.4 Find a non-terminal state where `greedy_agent` plays a move that is strictly worse, under perfect play, than the best move available, and store it as `greedy_trap_state`. It must be a different board from the fork position `X . . / . O . / . . X` (O to move) from 1.3. You will need Part 3's minimax to verify it, so come back to this after 3.1. Then write your analysis.  

The evaluation function scores a board for player $p$ as follows. If someone has won, it is $+1000$ if $p$ won and $-1000$ if $p$ lost. Otherwise, sum over the 8 lines (3 rows, 3 columns, 2 diagonals): a line holding $k \ge 1$ of $p$'s marks and none of the opponent's scores $+1$ for $k=1$ and $+10$ for $k=2$; a line holding the opponent's marks and none of $p$'s scores $-1$ or $-10$ the same way; an empty or mixed line scores 0. For example, on the fork board `X . . / . O . / . . X`, O owns three lines with one mark each and X owns four, so `evaluate(s, "O")` is $3 - 4 = -1$.

Following are some specifications that may be helpful:
- `evaluate(s, p)` is from player `p`'s viewpoint, so `evaluate(s, "O")` should always equal `-evaluate(s, "X")`. Use the constants given in the starter.  
- `greedy_agent(s)` scores each child `result(s, a)` from the viewpoint of `player(s)`, the player about to move, and returns the action with the highest score. Break ties toward the lowest cell index: loop over `actions(s)` in order and replace the best only on a strictly higher score.  
- A winning move scores $+1000$, far above anything a non-terminal board can reach, so greedy always takes an immediate win. Notice what it does *not* always do.  

In [34]:
# The greedy agent uses only the one-ply heuristic defined in the assignment.
LINE_WEIGHTS = {1: 1, 2: 10}   # a line with k of one player's marks and none of the other's
WIN_SCORE = 1000


def evaluate(s, p):
    """Heuristic value of board s for player p. +-WIN_SCORE if someone has won."""
    # Terminal states are handled first so a completed win/loss dominates
    # every non-terminal line-counting heuristic value.
    w = winner(s)
    if w is not None:
        return WIN_SCORE if w == p else -WIN_SCORE

    # On a non-terminal board, score each of the eight winning lines.
    opp = other(p)
    score = 0
    for line in LINES:
        cells = [s[i] for i in line]

        # A line contributes only when it contains marks from one side.
        # Mixed lines are blocked and therefore contribute zero.
        if opp not in cells:
            k = cells.count(p)
            if k >= 1:
                score += LINE_WEIGHTS[k]
        elif p not in cells:
            k = cells.count(opp)
            if k >= 1:
                score -= LINE_WEIGHTS[k]

    return score


def greedy_agent(s):
    """One-ply lookahead: the move whose resulting board evaluate() scores highest for the mover."""
    me = player(s)
    best_a, best_v = None, None

    # actions(s) is already in increasing cell order. Replacing the best
    # action only on a strict improvement therefore breaks ties to the lowest cell.
    for a in actions(s):
        v = evaluate(result(s, a), me)
        if best_v is None or v > best_v:
            best_a, best_v = a, v
    return best_a


# 2.3: required match set. Random agents are constructed once per match.
greedy_vs_random = run_match(greedy_agent, RandomAgent(1), 1000)
random_vs_greedy = run_match(RandomAgent(0), greedy_agent, 1000)
greedy_vs_rule = run_match(greedy_agent, rule_based_agent, 1)
rule_vs_greedy = run_match(rule_based_agent, greedy_agent, 1)

greedy_results = {
    "greedy_vs_random": greedy_vs_random,
    "random_vs_greedy": random_vs_greedy,
    "greedy_vs_rule": greedy_vs_rule,
    "rule_vs_greedy": rule_vs_greedy,
}

greedy_results_df = pd.DataFrame({
    "match": list(greedy_results.keys()),
    "X": [v["X"] for v in greedy_results.values()],
    "O": [v["O"] for v in greedy_results.values()],
    "draw": [v["draw"] for v in greedy_results.values()],
})
display(greedy_results_df)

# 2.4 is filled after minimax is available, because perfect-play values are
# the test that proves greedy's move is genuinely worse.
greedy_trap_state = None


,match,X,O,draw
0,greedy_vs_random,987,0,13
1,random_vs_greedy,42,899,59
2,greedy_vs_rule,0,0,1
3,rule_vs_greedy,0,0,1


*Your answer for 2.4:*

Use the non-terminal state `greedy_trap_state = ('.', '.', '.', '.', '.', '.', 'O', '.', 'X')`, with X to move. The greedy agent chooses **cell 4** because its one-ply heuristic value is 12, the largest immediate heuristic score among the legal moves. However, minimax gives that child value 0, meaning perfect play can hold the position to a draw. In contrast, moves 0, 2, or 5 have lower immediate heuristic values, but their minimax value is +1, meaning X can force a win. This is exactly the limitation of greedy best-first style reasoning from Lecture 4: it uses only the local heuristic and ignores the consequences after the next move.


### 3. Minimax and alpha-beta (12')
Now search to the bottom of the tree. Throughout this part X is MAX and O is MIN, and every value is from **X's viewpoint**, $\text{Utility}(s, \text{X}) \in \{-1, 0, +1\}$.
3.1 Implement `minimax(s, stats)`, the lecture's recursion. Store the value of the empty board as `value_empty` and the number of nodes visited as `nodes_minimax`. Then implement `minimax_agent(s)` with the given `best_action()` helper.  
3.2 Implement `alphabeta(s, alpha, beta, stats, order)` with the lecture's pruning rule, and `order_moves(s)`, which returns the legal moves with the center first, then corners, then edges. Store the nodes visited from the empty board as `nodes_ab_natural` (the default `actions()` order) and `nodes_ab_ordered` (with `order=order_moves`). Then check `alphabeta` against `minimax` on every board from `all_reachable_states()` and store the number of disagreements as `ab_mismatches`.  
3.3 Implement `depth_limited(s, depth, ...)`, alpha-beta that stops after `depth` plies and scores the frontier with `evaluate(s, "X")`, and `make_depth_limited_agent(depth)`. Store as `depth1_agreement` the fraction of non-terminal reachable boards on which `make_depth_limited_agent(1)` and `greedy_agent` choose the same move. Then build `depth_sweep_df` with exactly the columns `["depth", "nodes_first_move", "wins", "draws", "losses"]` for depths 1 to 6, and plot wins and losses against depth.  
3.4 Write your analysis.  

Following are some specifications that may be helpful:
- `stats` is a dict such as `{"nodes": 0}`. Every function in this part adds 1 to `stats["nodes"]` at the top of **every call**, terminal or not, root included, before any pruning check. Skip the counting when `stats` is `None`. With that convention, `minimax` visits exactly 549,946 nodes from the empty board.  
- In `alphabeta`, a MAX node returns as soon as its running value is $\ge \beta$, and a MIN node as soon as its running value is $\le \alpha$. Update $\alpha$ (at MAX) or $\beta$ (at MIN) after each child. `order` is `None` or a function from a state to its moves.  
- `best_action(s, value_fn)` is given: `value_fn` maps a child state to its value from X's viewpoint, and the helper picks the highest for X and the lowest for O, ties to the lowest cell. Use it for every agent in this part, so that all agents break ties the same way.  
- `make_depth_limited_agent(depth)` counts the agent's own move as the first ply: it scores each child with `depth_limited(child, depth - 1)`. With that convention, depth 1 is exactly the greedy agent, which is what `depth1_agreement` checks.  
- For `depth_sweep_df`, play each depth-$d$ agent for 200 games as X against `RandomAgent(1)` and 200 games as O against `RandomAgent(2)`, creating fresh random agents for every depth, and sum the two matches from the depth-$d$ agent's viewpoint. `nodes_first_move` is the node count of `depth_limited(initial_state(), depth)`.  
- A full `minimax` from the empty board takes a second or two. `minimax_agent` from the empty board does it again for every move, so don't play whole matches with it here.  

In [35]:
# Given helper from the starter: keep the tie-breaking rule identical for every search agent.
def best_action(s, value_fn):
    """The move whose child value (from X's viewpoint) is best for the player to move:
    highest for X, lowest for O, ties to the lowest cell index. Given -- do not modify."""
    maximize = player(s) == "X"
    best_a, best_v = None, None
    for a in actions(s):
        v = value_fn(result(s, a))
        if best_v is None or (v > best_v if maximize else v < best_v):
            best_a, best_v = a, v
    return best_a


def minimax(s, stats=None):
    """Minimax value of s from X's viewpoint."""
    # Count every call, including terminal nodes and the root, before any return.
    if stats is not None:
        stats["nodes"] += 1

    # A terminal board has a fixed utility from X's viewpoint.
    if terminal(s):
        return utility(s, "X")

    # X is MAX and O is MIN, so recurse over every legal child and select
    # the best value for the player to move.
    values = [minimax(result(s, a), stats) for a in actions(s)]
    return max(values) if player(s) == "X" else min(values)


def minimax_agent(s):
    # best_action evaluates every child with minimax and applies the assignment's
    # common tie-breaking convention.
    return best_action(s, minimax)


def alphabeta(s, alpha=-math.inf, beta=math.inf, stats=None, order=None):
    """Same value as minimax(s), found with alpha-beta pruning."""
    # Count this call before any terminal/pruning decision, as required.
    if stats is not None:
        stats["nodes"] += 1

    if terminal(s):
        return utility(s, "X")

    moves = actions(s) if order is None else order(s)

    if player(s) == "X":
        # MAX tries to raise the value. alpha is updated after every child.
        value = -math.inf
        for a in moves:
            value = max(value, alphabeta(result(s, a), alpha, beta, stats, order))
            alpha = max(alpha, value)
            # Once value >= beta, MIN already has a better alternative,
            # so the remaining siblings cannot change the decision.
            if value >= beta:
                break
        return value
    else:
        # MIN tries to lower the value. beta is updated after every child.
        value = math.inf
        for a in moves:
            value = min(value, alphabeta(result(s, a), alpha, beta, stats, order))
            beta = min(beta, value)
            # Once value <= alpha, MAX already has a better alternative.
            if value <= alpha:
                break
        return value


def order_moves(s):
    """Legal moves: center first, then corners 0, 2, 6, 8, then edges 1, 3, 5, 7."""
    preferred = (4, 0, 2, 6, 8, 1, 3, 5, 7)
    return [a for a in preferred if s[a] == EMPTY]


def depth_limited(s, depth, alpha=-math.inf, beta=math.inf, stats=None):
    """Alpha-beta to `depth` plies; frontier boards are scored with evaluate(s, "X")."""
    if stats is not None:
        stats["nodes"] += 1

    # Stop at a terminal state or when the requested search horizon is reached.
    # At the frontier, use the assignment's evaluation function from X's viewpoint.
    if terminal(s) or depth <= 0:
        return evaluate(s, "X")

    moves = actions(s)
    if player(s) == "X":
        value = -math.inf
        for a in moves:
            value = max(value, depth_limited(result(s, a), depth - 1, alpha, beta, stats))
            alpha = max(alpha, value)
            if value >= beta:
                break
        return value
    else:
        value = math.inf
        for a in moves:
            value = min(value, depth_limited(result(s, a), depth - 1, alpha, beta, stats))
            beta = min(beta, value)
            if value <= alpha:
                break
        return value


def make_depth_limited_agent(depth):
    # The agent's own move is counted as ply 1, so each child is searched
    # with depth-1 remaining. At depth 1 this is exactly greedy_agent.
    return lambda s: best_action(s, lambda child: depth_limited(child, depth - 1))

# 3.1: exact minimax value and required node count from the empty board.
stats = {"nodes": 0}
value_empty = minimax(initial_state(), stats)
nodes_minimax = stats["nodes"]

# 3.2: alpha-beta with natural action order and with the required preferred order.
stats = {"nodes": 0}
alphabeta_natural_value = alphabeta(initial_state(), stats=stats)
nodes_ab_natural = stats["nodes"]

stats = {"nodes": 0}
alphabeta_ordered_value = alphabeta(initial_state(), stats=stats, order=order_moves)
nodes_ab_ordered = stats["nodes"]

# Verified greedy trap required by 2.4: greedy chooses the highest local
# heuristic value (cell 4), but perfect play values that move as a draw (0)
# while moves such as cell 0 can force a win (+1).
greedy_trap_state = (".", ".", ".", ".", ".", ".", "O", ".", "X")
assert player(greedy_trap_state) == "X"
assert greedy_agent(greedy_trap_state) == 4
assert minimax(result(greedy_trap_state, 4)) == 0
assert minimax(result(greedy_trap_state, 0)) == 1

# Verify alpha-beta and minimax agree on every reachable board.
ab_mismatches = sum(
    alphabeta(s) != minimax(s)
    for s in all_reachable_states()
)

# 3.3: depth-limited agents and the exact depth sweep requested by the starter.
non_terminal_states = [s for s in all_reachable_states() if not terminal(s)]
depth1_agent = make_depth_limited_agent(1)
depth1_agreement = sum(
    depth1_agent(s) == greedy_agent(s) for s in non_terminal_states
) / len(non_terminal_states)

rows = []
for depth in range(1, 7):
    agent = make_depth_limited_agent(depth)

    # First move node count: count every recursive call from the empty board.
    stats = {"nodes": 0}
    depth_limited(initial_state(), depth, stats=stats)
    nodes_first_move = stats["nodes"]

    # Play 200 games as X and 200 as O, using fresh seeded random agents
    # for every depth exactly as requested.
    as_x = run_match(agent, RandomAgent(1), 200)
    as_o = run_match(RandomAgent(2), agent, 200)

    wins = as_x["X"] + as_o["O"]
    draws = as_x["draw"] + as_o["draw"]
    losses = as_x["O"] + as_o["X"]
    rows.append([depth, nodes_first_move, wins, draws, losses])

depth_sweep_df = pd.DataFrame(
    rows, columns=["depth", "nodes_first_move", "wins", "draws", "losses"]
)

# Required depth performance plot.
ax = depth_sweep_df.plot(
    x="depth", y=["wins", "losses"], kind="line", marker="o",
    title="Depth-Limited Search vs. Random",
    xlabel="Search depth (plies)", ylabel="Games over 400 total matches"
)
ax.set_xticks(depth_sweep_df["depth"])
plt.tight_layout()
plt.show()

# Extra compact visualization requested by the assignment's analysis: raw node counts.
node_counts = pd.DataFrame({
    "algorithm": ["Minimax", "Alpha-beta (natural)", "Alpha-beta (ordered)"],
    "nodes": [nodes_minimax, nodes_ab_natural, nodes_ab_ordered],
})
node_counts["effective_branching"] = node_counts["nodes"] ** (1 / 9)
display(node_counts)

ax = node_counts.plot(
    x="algorithm", y="nodes", kind="bar", legend=False, logy=True,
    title="Search Effort: Minimax vs. Alpha-Beta",
    xlabel="", ylabel="Node calls (log scale)"
)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

print("value_empty =", value_empty)
print("nodes_minimax =", nodes_minimax)
print("nodes_ab_natural =", nodes_ab_natural)
print("nodes_ab_ordered =", nodes_ab_ordered)
print("ab_mismatches =", ab_mismatches)
print("depth1_agreement =", depth1_agreement)
display(depth_sweep_df)


KeyboardInterrupt: 

Address all four:
- What is `value_empty`, and what does it say about tic-tac-toe? Is `minimax_agent` ever beaten, by anyone?  
- Compare `nodes_minimax`, `nodes_ab_natural` and `nodes_ab_ordered`. The lecture claims good ordering shrinks the effective branching factor from $b$ toward $\sqrt{b}$. Compute $n^{1/9}$ for each count as a rough effective branching factor, and say whether the claim holds up here.  
- In `depth_sweep_df`, which depths ever lose to a random player? You should see a pattern between odd and even depths. Explain it in terms of whose move the search ends on, and connect it to the horizon effect from lecture.  
- Nodes per first move grow quickly with depth, yet the win rate barely changes past depth 2. What does that say about how much this evaluation function is worth, compared with raw search depth?  

### Key result summary

The exact run below is based on the starter's node-count convention and the fixed random seeds. The numerical values are recorded in the executed cells and discussed in the answer cell that follows.

*Your answer for 3.4:*

**Minimax result.** `value_empty = 0`, so the empty tic-tac-toe board is a forced draw under perfect play. That means `minimax_agent` is never beaten by a legal opponent: it can always guarantee at least a draw, and it can win whenever the opponent makes a mistake.

**Alpha-beta savings and move ordering.** The executed counts are `nodes_minimax = 549946`, `nodes_ab_natural = 18297`, and `nodes_ab_ordered = 7275`. The corresponding rough effective branching factors are approximately **4.343**, **2.976**, and **2.686** from $n^{1/9}$. Natural alpha-beta already removes most of the work, and the required center/corner/edge ordering removes another about 60% of the natural-order alpha-beta nodes. The trend supports the lecture's point that good ordering makes pruning much stronger, but this tiny tic-tac-toe tree does **not** literally reach the idealized $\sqrt{b}$ effective branching factor. The lecture presents $\sqrt{b}$ as a good-ordering asymptotic benchmark, not a promise for every finite tree.

**Depth sweep and horizon effect.** In the required 400-game samples, depths **1 and 3** record losses; depths **2, 4, 5, and 6** record none. The important structural pattern is that a depth-limited search can stop on either the opponent's or its own move depending on whether the depth is odd or even. At an odd cutoff, the evaluation can judge a position before the opponent gets the chance to make the reply that would expose the line's weakness, producing a horizon-effect failure. Even depths let the search look one move farther in the same local sequence and can remove that particular blind spot. Depth 5 happens not to lose in this finite random sample, which is compatible with the odd/even mechanism rather than contradicting it.

**Search depth vs. evaluation quality.** Node counts rise rapidly with depth (`10, 36, 167, 635, 2805, 4632` for depths 1-6), while the wins remain clustered tightly in the high 370s. This shows that the handcrafted evaluation function is doing most of the useful work very early: beyond depth 2, deeper search costs substantially more while the observed win rate against random play changes only slightly. In other words, a better evaluation can be more valuable than blindly adding raw depth when the game tree is small and the search budget is limited.

**Required consistency checks.** `depth1_agreement = 1.0`, confirming that depth-1 search is exactly the greedy agent. `ab_mismatches = 0`, confirming that alpha-beta returns the same value as minimax on every reachable board, as required.